In [ ]:
import pandas as pd
from functions import *

In [ ]:
df = read_dataframe("war_economic_impact_dataset.csv")
df = format_columns(df)
df.info()

In [ ]:
#Delete columns we don't want 

df = df.drop(columns= ['status', 'pre_war_unemployment_',
   'during_war_unemployment_',                 
  'unemployment_spike_percentage_points',     
 'most_affected_sector',                     
 'youth_unemployment_change_',               
  'pre_war_poverty_rate_',                   
 'during_war_poverty_rate_',                 
 'extreme_poverty_rate_', 'households_fallen_into_poverty_estimate', 'inflation_rate_',                          
'currency_devaluation_',                    
'cost_of_war_usd',                          
'estimated_reconstruction_cost_usd',        
'informal_economy_size_pre_war_',           
'informal_economy_size_during_war_',        
'black_market_activity_level',              
'primary_black_market_goods',               
'currency_black_market_rate_gap_',          
'war_profiteering_documented'])


In [ ]:
df

In [ ]:
df = df[(df["start_year"] >= 1990)]

In [ ]:
df

In [ ]:
#Delete complete duplicates

df.duplicated().sum()
df = df.drop_duplicates()

#Check for null values
df = df.reset_index(drop=True)
df.isna().sum()

#Check for unique values in each column and ensure they're all the same

for column in df.select_dtypes(include="object"):
    print(f"\nColumn: {column}")
    print(df[column].unique())

#Ensure countries and regions follow the UN Region/Country Classification
#In this case Middle East is in the Asia Region and the Sub-region 'South Asia' is updated to its general region Asia
df["primary_country"] = df["primary_country"].replace({"DRC": "Democratic Republic of the Congo", "Russia":"Russian Federation"})

df["region"] = df["region"].replace({"Middle East": "Asia", "South Asia":"Asia"})

In [ ]:
#check datatypes are correct 

df.dtypes

In [ ]:
#Look at the statistics to understand if there are any outliers or extreme values 

df.describe()

In [ ]:
df.groupby("conflict_name").size().sort_values(ascending=False)

#Lets check if the number of rows per conflict_name --> there are too many rows per conflict, which means the dataset has many simulated economic scenarios per conflict
#So we need to check each conflict by country because a country can be involved in more than one conflict -- e.g. Israel is involved in two conflicts 

df.groupby("conflict_name")["primary_country"].unique()


df.groupby(["conflict_name","primary_country"]).size()

#We should filter per country per conflict to ensure unique results rather than repeated conflict-country pairs with different scenarios 
df_clean = (df.groupby(["conflict_name","primary_country"]).agg({
        "region":"first",
        "conflict_type":"first",
        "start_year":"first",
        "end_year":"first",
        "food_insecurity_rate_":"mean",
        "gdp_change_":"mean"
    })
    .reset_index()
)

df_clean

df_clean.sort_values(by="start_year")

In [ ]:
df_clean = df_clean.rename(columns={"food_insecurity_rate_": "food_insecurity_rate",
    "gdp_change_": "gdp_change"})
df_clean["conflict_type"] = df_clean["conflict_type"].replace({"Interstate/Counter-insurgency" : "Interstate War"})

df_clean.to_csv("conflict_clean.csv", index=False)

In [ ]:
df_clean.sort_values(by="start_year")